# Notebook 1: Random Forest Pollutant Importance

This notebook trains a Random Forest Regressor using pollutant features only to identify the strongest AQI contributing pollutant. It also exports model artifacts for backend use.

In [1]:
import json
from pathlib import Path

import joblib
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import train_test_split

In [2]:
PROJECT_ROOT = Path('..').resolve()
DATA_DIR = PROJECT_ROOT / 'data'
MODEL_DIR = PROJECT_ROOT / 'backend' / 'models'
ARTIFACT_DIR = PROJECT_ROOT / 'backend' / 'artifacts'
MODEL_DIR.mkdir(parents=True, exist_ok=True)
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

aqi_df = pd.read_csv(DATA_DIR / 'aqi_data.csv')
aqi_df = aqi_df.rename(columns={'S02': 'SO2'})
aqi_df.head()

,city_id,date_key,AQI,PM2.5,PM10,NO2,SO2,CO,O3,NH3,AQI Category
0,6,14-10,34,8.24,17.3,12.46,1.51,0.34,0.04,11.75,Good
1,41,14-10,34,8.24,17.3,12.46,1.51,0.34,0.04,11.75,Good
2,71,14-10,34,8.24,17.3,12.46,1.51,0.34,0.04,11.75,Good
3,126,14-10,34,8.24,17.3,12.46,1.51,0.34,0.04,11.75,Good
4,181,14-10,34,8.24,17.3,12.46,1.51,0.34,0.04,11.75,Good


In [3]:
pollutant_cols = ['PM2.5', 'PM10', 'NO2', 'SO2', 'CO', 'O3', 'NH3']
target_col = 'AQI'

model_df = aqi_df[pollutant_cols + [target_col]].copy()
for col in pollutant_cols + [target_col]:
    model_df[col] = pd.to_numeric(model_df[col], errors='coerce')

model_df = model_df.dropna(subset=pollutant_cols + [target_col])
X = model_df[pollutant_cols]
y = model_df[target_col]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

rf_model = RandomForestRegressor(
    n_estimators=100,
    max_depth=10,
    min_samples_split=5,
    min_samples_leaf=2,
    random_state=42,
    n_jobs=-1
)
rf_model.fit(X_train, y_train)

test_pred = rf_model.predict(X_test)
mae = float(mean_absolute_error(y_test, test_pred))
r2 = float(r2_score(y_test, test_pred))

print(f'MAE: {mae:.4f}')
print(f'R2 : {r2:.4f}')

MAE: 1.0776
R2 : 0.9463


In [4]:
importance_df = pd.DataFrame({
    'pollutant': pollutant_cols,
    'importance': rf_model.feature_importances_
}).sort_values('importance', ascending=False).reset_index(drop=True)

top_pollutant = importance_df.loc[0, 'pollutant']
importance_df

,pollutant,importance
0,PM2.5,0.881829
1,PM10,0.084972
2,CO,0.011321
3,NO2,0.009268
4,O3,0.005428
5,SO2,0.003673
6,NH3,0.003508


In [5]:
rf_model_path = MODEL_DIR / 'random_forest.pkl'
importance_csv_path = ARTIFACT_DIR / 'feature_importance.csv'
importance_json_path = ARTIFACT_DIR / 'feature_importance.json'

tmp_model_path = rf_model_path.with_suffix(rf_model_path.suffix + '.tmp')
tmp_csv_path = importance_csv_path.with_suffix(importance_csv_path.suffix + '.tmp')
tmp_json_path = importance_json_path.with_suffix(importance_json_path.suffix + '.tmp')

feature_importance_payload = {
    'model': 'RandomForestRegressor',
    'objective': 'AQI pollutant-level importance',
    'target': target_col,
    'features': pollutant_cols,
    'top_pollutant': top_pollutant,
    'metrics': {
        'mae': round(mae, 6),
        'r2': round(r2, 6)
    },
    'feature_importance': [
        {
            'pollutant': row['pollutant'],
            'importance': round(float(row['importance']), 8)
        }
        for _, row in importance_df.iterrows()
    ]
}

try:
    joblib.dump(rf_model, tmp_model_path)
    importance_df.to_csv(tmp_csv_path, index=False)
    with open(tmp_json_path, 'w', encoding='utf-8') as fp:
        json.dump(feature_importance_payload, fp, indent=2)

    tmp_model_path.replace(rf_model_path)
    tmp_csv_path.replace(importance_csv_path)
    tmp_json_path.replace(importance_json_path)
finally:
    for tmp_path in [tmp_model_path, tmp_csv_path, tmp_json_path]:
        if tmp_path.exists():
            tmp_path.unlink()

feature_importance_payload

{'model': 'RandomForestRegressor',
 'objective': 'AQI pollutant-level importance',
 'target': 'AQI',
 'features': ['PM2.5', 'PM10', 'NO2', 'SO2', 'CO', 'O3', 'NH3'],
 'top_pollutant': 'PM2.5',
 'metrics': {'mae': 1.077553, 'r2': 0.946325},
 'feature_importance': [{'pollutant': 'PM2.5', 'importance': 0.88182924},
  {'pollutant': 'PM10', 'importance': 0.08497228},
  {'pollutant': 'CO', 'importance': 0.0113214},
  {'pollutant': 'NO2', 'importance': 0.00926838},
  {'pollutant': 'O3', 'importance': 0.00542772},
  {'pollutant': 'SO2', 'importance': 0.00367269},
  {'pollutant': 'NH3', 'importance': 0.0035083}]}